# LAB17 · COMERCIAL ARAGONESA S.L.
## Tu pipeline, de punta a punta, en una sola tarde

**Sesión 10 · el último cuaderno del curso**

Hoy no se aprende una herramienta nueva: **se conectan todas.** La aplicación del cuadro de mando
se levanta **desde este mismo contenedor**, sin salir de Jupyter.

```
   tu Parquet -> DuckDB -> Plotly -> CUADRO DE MANDO
                    |                      |
                    |                      +-> la IA REDACTA -> tu AUDITAS
                    +-> la IA GENERA SQL -> DuckDB EJECUTA -> tu ANCLA VERIFICA
```

> **Las anclas de hoy:** `999.535` filas · `429.892.547,06 €` · ticket `430,09` · `10` ciudades.

---
## Paso 0 · El punto de partida: tu propio almacén · `BASE`

La primera celda crea la vista limpia **sobre el Parquet que escribiste tú** en el LAB10, no sobre
el CSV original. A partir de aquí, tu almacén ES la fuente.

In [ ]:
import duckdb, os

RUTA_PARQUET = '../datasets/salida/ventas_limpio.parquet/*.parquet'

try:
    duckdb.sql(f"CREATE OR REPLACE VIEW ventas_limpio AS "
               f"SELECT * FROM '{RUTA_PARQUET}'")
    duckdb.sql("SELECT COUNT(*) FROM ventas_limpio").fetchone()
    origen = "el Parquet maestro (tu LAB10)"
except Exception:
    # Plan B: LIMPIO-v1 sobre el CSV crudo. Mismos numeros, otro camino.
    duckdb.sql("""CREATE OR REPLACE VIEW ventas_limpio AS
    SELECT id_venta, fecha, id_cliente, id_producto, categoria,
           unidades, precio_unitario,
           CASE WHEN COALESCE(TRIM(ciudad), '') = '' THEN NULL
                ELSE UPPER(SUBSTR(TRIM(ciudad),1,1))
                     || LOWER(SUBSTR(TRIM(ciudad),2)) END AS ciudad,
           canal
    FROM '../datasets/ventas.csv'
    WHERE precio_unitario > 0""")
    origen = "el CSV crudo (plan B: revisa tu LAB10 esta noche)"

filas = duckdb.sql("SELECT COUNT(*) FROM ventas_limpio").fetchone()[0]
print(f"  Origen: {origen}")
print(f"  Filas : {filas}")
print("  ANCLA 999535 -> VALIDADO" if filas == 999535
      else "  NO CUADRA. Avisa antes de seguir.")

> 💡 Esa vista **no lee el CSV**: lee una carpeta de ficheros Parquet que escribió tu pipeline.
> **El dato ya no viene de fuera: viene de tu almacén.**

---
## Paso 1 · Los tres conjuntos del cuadro de mando · `BASE`

Un cuadro de mando necesita **tres formas**: un titular, un ranking y una evolución.

In [ ]:
import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

import os

os.makedirs('../datasets/salida', exist_ok=True)

duckdb.sql("""COPY (
    SELECT canal,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio GROUP BY canal ORDER BY facturacion DESC
) TO '../datasets/salida/kpi_canal.csv' (HEADER)""")

duckdb.sql("""COPY (
    SELECT ciudad,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio WHERE ciudad IS NOT NULL
    GROUP BY ciudad ORDER BY facturacion DESC
) TO '../datasets/salida/kpi_ciudad.csv' (HEADER)""")

duckdb.sql("""COPY (
    SELECT STRFTIME(fecha, '%Y-%m') AS mes,
           ROUND(SUM(unidades*precio_unitario), 2) AS facturacion,
           COUNT(*)                                AS operaciones
    FROM ventas_limpio GROUP BY mes ORDER BY mes
) TO '../datasets/salida/kpi_mes.csv' (HEADER)""")

for f in ['kpi_canal.csv', 'kpi_ciudad.csv', 'kpi_mes.csv']:
    n = duckdb.sql(f"SELECT COUNT(*) FROM '../datasets/salida/{f}'").fetchone()[0]
    print(f"  {f:18s} {n:3d} filas")

# Esperado: canal 3 - ciudad 10 - mes 12

⚠️ El `WHERE ciudad IS NOT NULL` **no es un adorno: es una decisión declarada.** Las 3.030 ventas
sin ciudad desaparecen de ese fichero. Dilo en tu informe.

In [ ]:
import duckdb

duckdb.sql("SELECT * FROM '../datasets/salida/kpi_canal.csv'").show()
duckdb.sql("SELECT * FROM '../datasets/salida/kpi_ciudad.csv' LIMIT 4").show()
duckdb.sql("SELECT * FROM '../datasets/salida/kpi_mes.csv'").show()

# Control: Zaragoza 146920182.71 - y apunta cual es TU mes pico

✍️ **¿Cuál es tu mes pico? ¿Coincide con lo que habrías apostado?**



---
---
# Paso 2 · El taller: AI Studio · `BASE`

Abre **`aistudio.google.com`**. No es otro chat: es **el taller con los mandos que el chat esconde**.

| Mando | Para qué | Hoy |
|---|---|---|
| Selector de modelo | con cuál trabajas | el que uses aquí es el que irá al código |
| **Temperatura** | cuánto arriesga el muestreo | **bájala a 0.2** |
| Salida estructurada | que un programa pueda consumirla | localízala |

> 🗣️ **Ni la temperatura cero garantiza determinismo.** Por eso la doctrina es **anclas y criterios**.

### Prototipa aquí el prompt del informe

```
ROL       eres analista de datos senior y escribes para direccion
CONTEXTO  (pega tus KPIs del paso 1)
TAREA     redacta 5 hallazgos, un parrafo cada uno, accionables
FORMATO   markdown, un titular en negrita por hallazgo
REGLA     usa EXCLUSIVAMENTE las cifras proporcionadas. No inventes ninguna
```

**Cuando funcione, se congela y pasa al código.** Es lo que hacen los equipos de verdad.

✍️ **¿Qué cambiaste entre el primer intento y el que te convenció?**



---
---
# Paso 3 · La IA dentro de tu pipeline · `BASE`

## 3.1 · La clave, con su liturgia

▸ **Terminal de Jupyter**: `export GEMINI_API_KEY=tu_clave` — y **reinicia el kernel**.

In [ ]:
import os

API_KEY = os.environ.get("GEMINI_API_KEY", "")
# API_KEY = "pega-aqui-tu-clave-SOLO-para-la-clase"   # <-- y BORRALA antes de entregar

print("Clave cargada." if API_KEY
      else "  FALTA la clave: export GEMINI_API_KEY=... y reinicia el kernel")

> ⚠️ La clave es una contraseña: **entorno sí, cuaderno entregado jamás.** La celda de entrega la
> busca y se niega a archivar si la encuentra.

## 3.2 · La llamada, por dentro

In [ ]:
import json, urllib.request, urllib.error

MODELO = "gemini-2.0-flash"     # <-- CONSTANTE: los nombres de modelo caducan (404)

def gemini(prompt, temperatura=0.2):
    url = (f"https://generativelanguage.googleapis.com/v1beta/models/"
           f"{MODELO}:generateContent?key={API_KEY}")
    cuerpo = {"contents": [{"parts": [{"text": prompt}]}],
              "generationConfig": {"temperature": temperatura}}
    peticion = urllib.request.Request(
        url, data=json.dumps(cuerpo).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST")
    try:
        with urllib.request.urlopen(peticion, timeout=60) as r:
            datos = json.load(r)          # <-- la respuesta ES JSON: tu LAB04, otra vez
        return datos["candidates"][0]["content"]["parts"][0]["text"]
    except urllib.error.HTTPError as e:
        return (f"[HTTP {e.code}]  429=cuota (espera 1 min)"
                f" · 403/400=la clave · 404=el modelo")

print(gemini("Responde solo con una palabra: capital de Aragon"))

> 💡 **La respuesta del servicio ES JSON**, y la navegas igual que tu `productos.json` del LAB04:
> `candidates` → `content` → `parts` → `text`. **Nada nuevo: lo mismo, en otro sitio.**

## 3.3 · ⭐ El patrón, verificado

In [ ]:
import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

def limpiar_sql(texto):
    """La IA envuelve el SQL en adornos. Higiene antes de ejecutar."""
    t = texto.strip()
    if t.startswith("```"):
        t = t.split("```")[1]
        if t.lower().startswith("sql"):
            t = t[3:]
    return t.strip().rstrip(";").strip()

ESQUEMA = duckdb.sql("DESCRIBE SELECT * FROM ventas_limpio").df().to_string(index=False)

CONTEXTO = f"""Eres un analista de datos senior.
Tengo una vista en DuckDB llamada exactamente: ventas_limpio
Su esquema es:
{ESQUEMA}
Responde SOLO con la consulta SQL, sin explicaciones y sin markdown."""

sql = limpiar_sql(gemini(
    CONTEXTO + "\nTarea: la facturacion total, es decir la suma de unidades por "
               "precio_unitario, redondeada a dos decimales."))

print("SQL GENERADO:\n", sql, "\n")

ANCLA = 429892547.06
try:
    obtenido = float(duckdb.sql(sql).fetchone()[0])
    print(f"  obtenido: {obtenido}")
    print("  ANCLA 429892547.06 -> VERIFICADO" if abs(obtenido - ANCLA) < 0.01
          else "  NO CUADRA -> a auditar el SQL. NO lo borres: leelo.")
except Exception as e:
    print(f"  El SQL no ejecuta: {type(e).__name__}. Eso TAMBIEN es un hallazgo.")

### ✍️ LA AUDITORÍA — esto es lo que se entrega

**Si salió `VERIFICADO`:** ¿qué parte del contexto se lo puso fácil?
**Si salió `NO CUADRA`:** **no lo borres.** Busca **la decisión que el modelo tomó por ti**:
¿olvidó el `ROUND`? ¿contó los `NULL`? ¿se inventó una columna?

✍️



---
---
# Paso 4 · El cuadro de mando · `BASE`
## La aplicación corre en este mismo contenedor

### 4.1 · Las dependencias

Solo faltan dos paquetes. `duckdb` ya lo tienes desde la sesión 4.

In [ ]:
# starlette 1.4 anadio un argumento OBLIGATORIO que la version actual de
# streamlit no le pasa (GZipResponder: thread_minimum_size). El servidor arranca,
# pero el NAVEGADOR recibe un 500 en cuanto pide compresion. Se fija la version.
%pip install -q streamlit plotly "starlette<1.4"

import importlib
for m in ("streamlit", "plotly", "duckdb"):
    try:
        print(f"  {m:10s} {importlib.import_module(m).__version__}")
    except Exception as e:
        print(f"  {m:10s} FALTA -> {e}")

> ⚠️ **Si acabas de instalarlos, no hace falta reiniciar el kernel**: la app corre en un proceso
> aparte, no dentro de este.

### 4.2 · Comprobar que `app.py` está donde toca

In [ ]:
import duckdb

# La vista vive en el KERNEL, no en el disco: si lo has reiniciado, se rehace
# sola sobre tu Parquet. Si tampoco está el Parquet, ejecuta el Paso 0.
try:
    duckdb.sql("SELECT 1 FROM ventas_limpio LIMIT 1")
except duckdb.CatalogException:
    duckdb.sql("CREATE OR REPLACE VIEW ventas_limpio AS SELECT * FROM "
               "'../datasets/salida/ventas_limpio.parquet/*.parquet'")

import os

APP = "app.py"
estado = "encontrado" if os.path.exists(APP) else "NO ESTA — pideselo al docente"
print(f"  {APP}: {estado}")
if os.path.exists(APP):
    n_lineas = sum(1 for _ in open(APP, encoding="utf-8"))
    print(f"  {n_lineas} lineas · leelas: no hay magia dentro")

# La app necesita esto:
NECESITA = ["../datasets/salida/ventas_limpio.parquet",
            "../datasets/salida/kpi_canal.csv"]
for f in NECESITA:
    print(f"  {'OK  ' if os.path.exists(f) else 'FALTA'} {f}")

### 4.3 · Arrancar la aplicación

La lanzamos **en segundo plano** desde el propio cuaderno. Así la terminal te queda libre — y
aprendes que un servidor es un proceso más, no una ventana.

In [ ]:
import os

import subprocess, sys, time, socket

LOG = "streamlit.log"
PUERTO = 8501

def puerto_ocupado(p):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", p)) == 0

if puerto_ocupado(PUERTO):
    print(f"  Ya hay algo escuchando en :{PUERTO}."
          "  Si es tu app, no la arranques dos veces.")
else:
    proceso = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", APP,
         "--server.port", str(PUERTO), "--server.address", "0.0.0.0",
         "--server.headless", "true", "--browser.gatherUsageStats", "false"],
        stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
        env={**os.environ, "GEMINI_API_KEY": API_KEY})
    print(f"  Lanzada (pid {proceso.pid}). Registro en {LOG}")
    time.sleep(6)

### 4.4 · ¿Responde? El semáforo antes de abrir el navegador

In [ ]:
import urllib.request, time

# Se pide la PAGINA con compresion, como haria un navegador. Preguntar solo por
# /_stcore/health no vale: el servidor puede estar vivo y la pagina dar un 500.
def como_un_navegador(url):
    p = urllib.request.Request(url)
    p.add_header("Accept-Encoding", "gzip, deflate")
    return urllib.request.urlopen(p, timeout=5).status

for intento in range(12):
    try:
        if como_un_navegador(f"http://127.0.0.1:{PUERTO}/") == 200:
            print("  La aplicacion RESPONDE, y sirve la pagina.")
            break
    except urllib.error.HTTPError as e:
        print(f"  El servidor vive pero la pagina da {e.code}. Mira el registro:")
        print("".join(open(LOG, encoding="utf-8", errors="ignore").readlines()[-14:]))
        break
    except Exception:
        time.sleep(2)
else:
    print("  No responde todavia. Mira el registro:")
    print("".join(open(LOG, encoding="utf-8", errors="ignore").readlines()[-14:]))

### 4.5 · Abrirla

**Coge la URL con la que abriste JupyterLab y cambia `:8888` por `:8501`.**

```
   http://IP-DE-TU-VM:8501
```

> ⚠️ **La IP de la VM, no `localhost`.** Tu `localhost` es tu Windows; el servicio corre en la VM.
> Tercera vez en el curso: `:8888`, `:9870`, y ahora `:8501`. **Ya deberías verla venir.**
>
> 💡 **Si el navegador no carga nada**, el puerto no está publicado en el compose. Avisa: se
> arregla en el host con `aula/docker-compose.override.yml` (tu docente lo tiene).

### 4.6 · Lo que hay que mirar · `BASE`

| Zona | La pregunta |
|---|---|
| La banda del ancla | ¿qué pasaría si saliera roja? ¿se puede mirar el resto? |
| Los cuatro KPIs | quita un canal: **¿qué número cambia y cuál no?** |
| «Solo con ciudad conocida» | se van 3.030 ventas. **¿Eso es limpiar o es esconder?** |
| El mes pico | ¿campaña, estacionalidad, o un artefacto de los datos? |
| «Ver los datos en crudo» | **todo gráfico debe poder abrirse** |

✍️ **Una cosa que hayas descubierto moviendo filtros y que no supieras antes:**


### 4.7 · ⭐ El botón del informe, y su auditoría · `BASE`

Pulsa **Generar informe**. La IA redacta **a partir de KPIs que ya pasaron por tu ancla**: no
calcula nada, y ese es el diseño. Abre «ver el prompt exacto» y **busca las cinco piezas dentro**.

| Comprobación | ✔ |
|---|---|
| ¿Cita **solo** las cifras que la app le dio? | |
| ¿Alguna cifra **inventada** o redondeada raro? | |
| ¿Algún hallazgo que **los datos no sostienen**? | |
| ¿Lo firmarías y se lo darías a Dirección? | |

✍️ *(tu veredicto, con el número concreto si encontraste algo)*


### 4.8 · Parar la aplicación *(al terminar la clase)*

In [ ]:
import signal

try:
    proceso.send_signal(signal.SIGTERM)
    print("  Aplicacion detenida.")
except NameError:
    print("  No la habias arrancado desde este cuaderno.")

---
---
# Paso 5 · ⭐ TU MEJORA · `COMPLETA`
## La aplicación funciona. Lo que le falta lo decides tú.

Hasta aquí has ejecutado lo que otro escribió. **Este paso es al revés:** eliges qué le falta a la
aplicación, se lo pides a una IA, y **decides si lo que te devuelve sirve**.

No se puntúa que la IA escriba código. Se puntúa **que tú sepas si el código es correcto** — y para
eso hace falta lo del curso entero: conocer tus tablas, y tener un número contra el que contrastar.

## Elige UNA · el detalle está en `MEJORAS.md`

| | Mejora | De qué va | Su ancla |
|---|---|---|---|
| **⓵** | **Cobertura de stock** | Cruzar ventas con catálogo: ¿cuántos meses aguanta el stock de lo que más se vende? | El nº1 por facturación es el **Monitor 27 QHD Compact**, 14,54 M€ |
| **⓶** | **Evolución por segmento** | Abrir la línea mensual en tres, una por segmento de cliente | Las operaciones de los tres segmentos suman **999.535** |
| **⓷** | **Descargar lo filtrado** | Un botón que exporte a CSV exactamente lo que el filtro deja pasar | El CSV tiene **las mismas filas que dice el marcador** |

✍️ **La que eliges, y por qué:**



## 5.1 · El criterio, ANTES de pedir nada · `COMPLETA`

Escribe **qué vas a comprobar** para dar la mejora por buena. Antes de ver una sola línea de código.

> 🗣️ ¿Por qué antes? Porque después **el sesgo elige**: damos por bueno lo que suena seguro y lo que
> nos ha costado conseguir. Es la misma regla de tus predicciones de todo el curso, aplicada a ti.

✍️ **Mi criterio de éxito:**

1.
2.
3.



## 5.2 · El prompt · `COMPLETA`

Las cinco piezas de siempre. **Adjunta `GUIA_PROGRAMACION_RAG.md` y `app.py`**: sin ese contexto, el
modelo se inventará nombres de columna y se olvidará del alias en los JOIN.

```
ROL       eres desarrollador Python senior sobre una app Streamlit + DuckDB
CONTEXTO  (adjunta GUIA_PROGRAMACION_RAG.md y app.py)
TAREA     escribe la funcion pagina_<lo que sea>(filtro) que ...
FORMATO   solo el codigo de la funcion y su linea del diccionario PAGINAS
EJEMPLO   (pega pagina_productos entera, del app.py)
```

✍️ **Pega aquí el prompt que has usado, entero:**



## 5.3 · Prueba el SQL AQUÍ antes de tocar `app.py` · `COMPLETA`

**No pegues el código en la aplicación todavía.** La consulta que te haya escrito la IA se prueba
primero en el cuaderno: si está mal, aquí lo ves en tres segundos; dentro de la app, ves una
pantalla en blanco y no sabes por qué.

**Copia la consulta de tu función, pégala abajo y ejecútala.**

In [ ]:
import duckdb

# Las tres vistas, como en la app. Se rehacen solas si reiniciaste el kernel.
duckdb.sql("CREATE OR REPLACE VIEW clientes AS "
           "SELECT * FROM '../datasets/clientes.csv'")
duckdb.sql("""CREATE OR REPLACE VIEW productos AS
    SELECT id, nombre, categoria, precio,
           stock.central + stock.tiendas AS stock_total
    FROM read_json('../datasets/productos.json')""")

# ── PEGA AQUI la consulta de tu mejora, sin el f-string ni el filtro ────────
MI_CONSULTA = """
SELECT p.nombre, p.stock_total, COALESCE(SUM(v.unidades), 0) AS vendidas
FROM productos p
LEFT JOIN ventas_limpio v ON v.id_producto = p.id
GROUP BY p.nombre, p.stock_total
ORDER BY vendidas DESC
LIMIT 5
"""

duckdb.sql(MI_CONSULTA).show()

> ⚠️ **`Ambiguous reference to column name`** significa que dos tablas tienen esa columna: hay que
> apellidarla (`v.categoria` o `p.categoria`). Es el fallo número uno cuando la IA escribe un JOIN,
> y está avisado en la guía.

## 5.4 · Contrasta contra el ancla de tu mejora · `COMPLETA`

Tu consulta ya devuelve algo. **Que devuelva algo no es que esté bien.**

In [ ]:
import duckdb

# El ancla de CADA mejora. Descomenta la tuya.
# ⓵ COBERTURA: el nº1 por facturacion tiene que ser el Monitor 27 QHD Compact
# duckdb.sql("""SELECT p.nombre,
#         ROUND(SUM(v.unidades*v.precio_unitario)/1e6, 2) AS millones
#     FROM productos p JOIN ventas_limpio v ON v.id_producto = p.id
#     GROUP BY p.nombre ORDER BY millones DESC LIMIT 1""").show()
# ⓶ SEGMENTOS: las operaciones de los tres tienen que sumar 999.535
# suma = duckdb.sql("""SELECT SUM(n) FROM (
#         SELECT COUNT(*) AS n FROM ventas_limpio v
#         JOIN clientes c ON c.id_cliente = v.id_cliente
#         GROUP BY c.segmento)""").fetchone()[0]
# print(f"  suma {suma}  ->  {'CUADRA' if suma == 999535 else 'EL JOIN MULTIPLICA'}")
# ⓷ DESCARGA: las filas del CSV tienen que ser las del marcador
# n = duckdb.sql("SELECT COUNT(*) FROM ventas_limpio").fetchone()[0]
# print(f"  el marcador dice {n}. Abre el CSV descargado y cuenta sus lineas.")

print("Descomenta el ancla de la mejora que hayas elegido.")

✍️ **El número del ancla y tu veredicto:**



## 5.5 · La auditoría · `COMPLETA`
### Es lo que se puntúa. Cinco minutos, en serio.

| Pregunta | ✍️ |
|---|---|
| ¿Usó **solo columnas que existen**? | |
| ¿**Cualificó el filtro** con el alias, al haber JOIN? | |
| ¿Qué **decidió por su cuenta** que tú no le habías pedido? | |
| ¿Cuadró el ancla **a la primera**? Si no, ¿qué reformulaste? | |

✍️ **Tu auditoría, en cuatro o cinco líneas:**


> 🎯 **Si te funcionó a la primera, no te quedes ahí.** Escribe **qué parte del contexto se lo puso
> fácil**: ¿la guía? ¿el ejemplo? ¿los nombres de las columnas? **Eso es lo que se transfiere al
> siguiente problema**, y puntúa igual.

## 5.6 · Llévalo a la aplicación

1. Abre `app.py` en JupyterLab y pega tu función **en la sección 3**.
2. Añade su línea al diccionario **`PAGINAS`**, al final del fichero.
3. Guarda. La aplicación se recarga sola; si no, botón **Rerun** arriba a la derecha.
4. **Captura de tu pestaña nueva.** Va en el entregable.

> ⚠️ Si al guardar la aplicación se cae, mira la terminal: el error está ahí, con su número de
> línea. **Cae al guardar, no al arrancar** — es la ventaja de tenerla corriendo mientras editas.

✍️ **¿Qué has tenido que tocar del código que te dio la IA para que encajara?**



---
---
# Paso 6 · 🏁 EL RETO · todo tu almacén en Parquet · `RETO`

Tu aplicación lee **tres cosas en tres formatos distintos**: las ventas en Parquet, los clientes en
CSV y los productos en JSON. Funciona — pero es exactamente el desorden con el que empezó el curso.

**El reto: dejarlo todo en el formato del presente, y medir lo que se gana.**

> 🗣️ Es el LAB05 otra vez, pero ahora sobre **tus** tablas y con **tu** aplicación delante. Y hay
> una recompensa inmediata: **la app lo detecta sola** y lo dice en la barra lateral.

## 6.1 · Convierte clientes y productos

Fíjate en lo que hace el `COPY` del segundo: **aplana el JSON anidado** —`stock.central` y
`stock.tiendas` sumados en una columna— antes de escribirlo. Es tu `jq` de la sesión 3, congelado en
un fichero.

In [ ]:
import duckdb, os

SALIDA = "../datasets/salida"
os.makedirs(SALIDA, exist_ok=True)

duckdb.sql(f"""COPY (SELECT * FROM '../datasets/clientes.csv')
TO '{SALIDA}/clientes.parquet' (FORMAT PARQUET)""")

duckdb.sql(f"""COPY (
    SELECT id, nombre, categoria, precio,
           stock.central + stock.tiendas AS stock_total
    FROM read_json('../datasets/productos.json')
) TO '{SALIDA}/productos.parquet' (FORMAT PARQUET)""")

print("  clientes.parquet y productos.parquet escritos")

## 6.2 · Y los KPIs, que también son datos

Los tres CSV que publicaste son tablas pequeñas: en Parquet ganarán poco. **Hazlo igualmente y mira
el número** — que un formato sea mejor no significa que lo sea siempre, y eso también se mide.

In [ ]:
import duckdb

SALIDA = "../datasets/salida"

for nombre in ("kpi_canal", "kpi_ciudad", "kpi_mes"):
    duckdb.sql(f"""COPY (SELECT * FROM '{SALIDA}/{nombre}.csv')
    TO '{SALIDA}/{nombre}.parquet' (FORMAT PARQUET)""")
    print(f"  {nombre}.parquet escrito")

## 6.3 · La medición · `RETO`

La misma pregunta del LAB05: **¿cuántas veces más pequeño?**

In [ ]:
import os

SALIDA = "../datasets/salida"

def kb(ruta):
    if os.path.isdir(ruta):
        return sum(os.path.getsize(os.path.join(ruta, f))
                   for f in os.listdir(ruta)) / 1024
    return os.path.getsize(ruta) / 1024

PAREJAS = [("../datasets/clientes.csv", f"{SALIDA}/clientes.parquet"),
           ("../datasets/productos.json", f"{SALIDA}/productos.parquet"),
           (f"{SALIDA}/kpi_canal.csv", f"{SALIDA}/kpi_canal.parquet"),
           (f"{SALIDA}/kpi_ciudad.csv", f"{SALIDA}/kpi_ciudad.parquet"),
           (f"{SALIDA}/kpi_mes.csv", f"{SALIDA}/kpi_mes.parquet")]

print(f"  {'original':26} {'KB':>9} {'parquet KB':>11} {'veces':>7}")
for a, b in PAREJAS:
    if os.path.exists(a) and os.path.exists(b):
        x, y = kb(a), kb(b)
        print(f"  {os.path.basename(a):26} {x:9.1f} {y:11.1f} {x/y:7.2f}")

✍️ **¿Cuál gana más y cuál casi nada? ¿Por qué?**


> 🎯 **La respuesta interesante no es «Parquet gana».** Es *cuándo* gana: con muchas filas y columnas
> repetidas. Un CSV de tres filas en Parquet puede llegar a ocupar **más**, por la cabecera del
> propio formato. **Un formato no es mejor: es mejor PARA algo.**

## 6.4 · Comprueba que no has perdido nada

Convertir no vale de nada si el dato cambia. **Contrasta las dos vías.**

In [ ]:
import duckdb

SALIDA = "../datasets/salida"

pares = [("clientes", "'../datasets/clientes.csv'", f"'{SALIDA}/clientes.parquet'")]
for etiqueta, uno_, dos_ in pares:
    a = duckdb.sql(f"SELECT COUNT(*) FROM {uno_}").fetchone()[0]
    b = duckdb.sql(f"SELECT COUNT(*) FROM {dos_}").fetchone()[0]
    veredicto = "IDENTICO" if a == b else "NO CUADRA"
    print(f"  {etiqueta}: original {a} · parquet {b} -> {veredicto}")

# Y el que de verdad importa: el stock total, por las dos vias
v = duckdb.sql("""SELECT SUM(stock.central + stock.tiendas)
    FROM read_json('../datasets/productos.json')""").fetchone()[0]
p = duckdb.sql(
    f"SELECT SUM(stock_total) FROM '{SALIDA}/productos.parquet'"
).fetchone()[0]
veredicto = "IDENTICO" if v == p else "NO CUADRA"
print(f"  stock total: JSON {v} · parquet {p} -> {veredicto}")

## 6.5 · Y ahora mira tu aplicación

**Recárgala** (botón *Rerun*, o para y vuelve a arrancar). Sin tocar una línea de código:

| Dónde | Qué debe decir ahora |
|---|---|
| Barra lateral | **Clientes: Parquet · Productos: Parquet** |
| Pestaña **Publicación** | La tabla de CSV contra Parquet, con tus veces |
| Pestaña **Publicación** | La verificación cruzada de los KPIs, **en verde** |

> 🎯 **La app no ha cambiado: han cambiado los datos, y ella se ha dado cuenta.** Eso es lo que
> significa que una aplicación esté bien hecha — y lo has conseguido con dos `COPY`.

✍️ **¿Qué dice la barra lateral, y qué dice la verificación cruzada?**



---
---
# 🔍 CONSULTA · Bloque final

**Cuatro preguntas, una de cada etiqueta. Se contestan hoy, en clase.**

**F1 · 🗂️ FUENTES** — *Según el material del curso, ¿por qué se eligió una **vista** y no otro fichero
para definir LIMPIO-v1? **Cítame el apartado.***

✍️


**F2 · ⚙️ MÁQUINA** *(esto no se le pregunta a nadie: se ejecuta)*

✍️ Tus números de hoy: filas · facturación · ticket · ciudades · mes pico · el veredicto del ancla.


**F3 · 🤖 ASISTENTE** — *Tengo un cuadro de mando con facturación por canal, por ciudad y por mes.
Propón **dos indicadores que NO tengo** y di qué decisión permitiría tomar cada uno.*
**Audítala:** ¿propone algo que tus datos no sostienen?

✍️


**F4 · 📝 CRITERIO** *(lo único que la IA no puede poner)*

✍️ ¿Qué pieza de esta semana llevarías mañana a tu puesto, y qué te falta para hacerlo?



---
---
# 📦 ENTREGABLE · CIERRE DEL CURSO

**`Ctrl+S` antes de nada.** Las celdas copian el fichero **guardado en disco**.

| # | Lo que tiene que estar | ¿Hecho? |
|---|---|---|
| 1 | El paso 0 en **VALIDADO** con 999.535 | |
| 2 | Los tres CSV publicados (3 · 10 · 12 filas) | |
| 3 | El patrón del paso 3.3 **y su auditoría escrita** | |
| 4 | El cuadro de mando visto, y **la auditoría del informe** | |
| 5 | **Tu mejora**: criterio, prompt, ancla contrastada y auditoría | |
| 6 | La **captura** de tu pestaña nueva | |
| 7 | *(reto)* Todo en Parquet, **medido y contrastado** | |
| 8 | El bloque de consulta contestado | |

## ① Antes de nada: ¿viaja alguna clave?

In [ ]:
import json

import glob, re

sospechosos = []
for ruta in glob.glob("*lab17*.ipynb"):
    if ".ipynb_checkpoints" in ruta:
        continue
    nb = json.load(open(ruta, encoding="utf-8"))
    for celda in nb["cells"]:
        for linea in "".join(celda["source"]).split("\n"):
            if linea.strip().startswith("#"):
                continue
            if re.search(r"AIza[0-9A-Za-z_\-]{20,}", linea):
                sospechosos.append((ruta, linea.strip()[:60]))

if sospechosos:
    print("  " + "=" * 66)
    print("   PARA. Hay algo que parece una clave dentro del cuaderno:")
    for r, l in sospechosos:
        print(f"     {r}:  {l}")
    print("   Borrala, Ctrl+S, y repite esta celda. (Y revocala en aistudio.)")
    print("  " + "=" * 66)
else:
    print("  Sin claves a la vista. Puedes archivar.")

## ② Archivar el cuaderno

In [ ]:
import glob
import json
import os

import shutil

SESION = 10

def resultados_en_disco(ruta):
    try:
        nb = json.load(open(ruta, encoding="utf-8"))
    except Exception:
        return 0, 0
    codigo = [c for c in nb["cells"] if c["cell_type"] == "code"]
    return sum(1 for c in codigo if c.get("outputs")), len(codigo)

os.makedirs("entregables", exist_ok=True)
cuadernos = [f for f in glob.glob("*lab17*.ipynb") if ".ipynb_checkpoints" not in f]

if not cuadernos:
    print("  PARA. No encuentro ningun cuaderno *lab17*.ipynb en esta carpeta.")
else:
    for c in cuadernos:
        con_salida, total = resultados_en_disco(c)
        if con_salida == 0:
            print(f"  PARA. {c} no tiene NI UN resultado guardado en disco.")
            print("        Pulsa Ctrl+S y repite esta celda.")
        else:
            destino = f"entregables/S{SESION:02d}_lab17.ipynb"
            shutil.copy(c, destino)
            print(f"  Archivado: {destino}"
                  f"   ({con_salida}/{total} celdas con resultado)")

## ③ Exportar a HTML

In [ ]:
import glob
import os

for c in glob.glob(f"entregables/S{SESION:02d}_*.ipynb"):
    salida = os.path.basename(c).replace(".ipynb", ".html")
    codigo = os.system(f'jupyter nbconvert --to html "{c}" --output "{salida}" '
                       f'--output-dir entregables 2>/dev/null')
    print(f"  {salida}  ->  {'OK' if codigo == 0 else 'FALLO'}")

## ④ El control que de verdad importa: contar los `[n]:`

In [ ]:
import glob
import os
import re

for h in sorted(glob.glob("entregables/*.html")):
    texto = open(h, encoding="utf-8", errors="ignore").read()
    n = len(re.findall(r"\[[0-9]+\]:", texto))
    print(f"  {os.path.basename(h):34s} {n:3d} celdas ejecutadas "
          f"{'' if n else '  <-- VACIO. Ctrl+S y repite.'}")

## ⑤ El paquete final del curso

In [ ]:
import glob
import os
import re

import zipfile

APELLIDO_NOMBRE = "PEREZ_Ana"        # <-- pon el tuyo ANTES de ejecutar

vacios = [os.path.basename(h) for h in glob.glob("entregables/*.html")
          if not re.findall(r"\[[0-9]+\]:",
                            open(h, encoding="utf-8", errors="ignore").read())]

if vacios:
    print("  NO EMPAQUETO: estos HTML no llevan resultados ->", ", ".join(vacios))
    print("  Ctrl+S y repite las celdas anteriores.")
else:
    nombre_zip = f"{APELLIDO_NOMBRE}_cierre.zip"
    with zipfile.ZipFile(nombre_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for raiz, carpetas, ficheros in os.walk("entregables"):
            carpetas[:] = [d for d in carpetas if d != ".ipynb_checkpoints"]
            for f in ficheros:
                z.write(os.path.join(raiz, f))
        for f in glob.glob("../datasets/salida/*.csv"):
            z.write(f, os.path.join("salida", os.path.basename(f)))
    tam = os.path.getsize(nombre_zip) / 1024
    print(f"  {nombre_zip}  ({tam:.0f} KB)  ->  subelo a Moodle")

---
---
# 🏁 La frase de salida — dila entera, en voz alta

> *«He construido un pipeline de datos real de punta a punta, con la IA como componente auditado.
> He tocado herramientas reales y las he combinado. Debo profundizar — y sé exactamente por dónde.
> **La base la tengo.»***

**Publicado en Moodle:** el **anexo del ecosistema** y el **bloque extra de continuidad** con su
proyecto. No son deberes: son la puerta siguiente.